In [ ]:
import kaggle_benchmarks as kbench

@kbench.task(name="generate_conclusion_from_result")
def conclusion_from_result(
    llm,
    result="Analysis of the quarterly data indicates a 15% growth in market share, primarily driven by the new product line launched in Q2.",
    conclusion="The new product line was a key factor in the company's market share growth for the quarter."
):
    prompt = f"From the following result statement, generate a concise, one-sentence conclusion:\n\nResult: \"{result}\""
    generated_conclusion = llm.prompt(prompt)

    assessment = kbench.assertions.assess_response_with_judge(
        criteria=[
            f"The generated conclusion must be factually consistent with this statement: '{conclusion}'"
        ],
        response_text=generated_conclusion,
        judge_llm=kbench.judge_llm
    )

    if assessment is None:
        kbench.assertions.assert_fail(expectation="Judge LLM failed to return a valid assessment.")
        # Ensure it returns the expected dictionary structure even on failure
        return {"passed": False, "reason": "Judge LLM failed to return a valid assessment."}
    
    # FIX: Track the overall outcome and rename loop variable to avoid overwriting 'result'
    passed = None
    reason = None
    
    for judge_result in assessment.results:
        kbench.assertions.assert_true(
            judge_result.passed,
            expectation=f"Factual consistency check failed. Reason: {judge_result.reason}"
        )
        passed=judge_result.passed
        reason=judge_result.reason
        # if not judge_result.passed:
        #     all_passed = False
        #     fail_reason = judge_result.reason
            
    return {
        "passed": passed,
        "reason": reason,
        "generated_text": generated_conclusion
    }

# Run a quick test on the first task
# o1 = conclusion_from_result.run(kbench.llm)
# o1.assertion_results[0]#.as_dataframe()


In [ ]:
n_jobs=5
name=["biorxiv","medrxiv"][0]
import os
import pandas as pd
file_csv = f"{name}.csv"
dataset_name = f"vmorozov/{name}-conclusions"
file_path = f"/kaggle/input/datasets/vmorozov/{name}-conclusions/{file_csv}" 

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
else:
    # Fallback for local testing or if you still want to use kagglehub interactively
    import kagglehub
    from kagglehub import KaggleDatasetAdapter
    df = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS,
        dataset_name,
        file_csv
    )
df=df.sample(10, random_state=42) #only for development, remove for final version
df_doi=df[['doi','journal']]

        


In [ ]:
@kbench.task(name="batch_conclusion_from_result")
def score_conclusion_from_result(llm, df) -> float:
    with kbench.client.enable_cache():
        runs = conclusion_from_result.evaluate(
            stop_condition=lambda runs: len(runs) == df.shape[0],  
            max_attempts=1, 
            llm=[llm], 
            evaluation_data=df[['result','conclusion']],
            n_jobs=n_jobs, 
        )

    eval_df = runs.as_dataframe()
    
    # Calculate accuracy safely
    accuracy = float(eval_df['result'].apply(lambda x: x.get("passed", False) if isinstance(x, dict) else x).mean())
    
    # ONLY return the float
    return accuracy 

# Run the scoring task safely
eval1 = score_conclusion_from_result.run(kbench.llm, df)
print(f"Benchmark Accuracy: {eval1.result}")

In [ ]:
# In a new notebook cell, run the evaluation directly to get the dataframe
with kbench.client.enable_cache():
    debug_runs = conclusion_from_result.evaluate(
        stop_condition=lambda runs: len(runs) == df.shape[0], # run on 5 rows
        max_attempts=1, 
        llm=[kbench.llm], 
        evaluation_data=df[['result','conclusion']],
        n_jobs=5, 
    )

# Extract the dataframe
results_df = debug_runs.as_dataframe()

# Create a clean column for the generated text
results_df['generated_conclusion'] = results_df['result'].apply(
    lambda x: x.get("generated_text") if isinstance(x, dict) else None
)

# Create a clean column for the pass/fail boolean
results_df['passed'] = results_df['result'].apply(
    lambda x: x.get("passed", False) if isinstance(x, dict) else x
)

# View the original conclusion, the generated one, and whether it passed
print(results_df[['conclusion', 'generated_conclusion', 'passed']])

In [ ]:
file_result='results_df.csv'
results=df_doi.reset_index().rename(columns={'index': 'id'}).merge(debug_runs.as_dataframe(),on='id')
results.to_csv(file_result,index=None)

In [ ]:
if False:
    import pandas as pd
    import ast
    results=pd.read_csv('out/conclusion-consistent/20260402_2012/results_df.csv')
    results['result'] = results['result'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [ ]:
print(results['result'].apply(lambda x: x['passed']).groupby(results['journal']).agg(['count', 'mean'])
)
